# Phase 5: Test Set Rebuild
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it) and `./local-data/phase4_training_pairs/<run id>/training_pairs.jsonl` (the Phase 4 training pairs, for the contamination guard). It uses the last run folder of each in name order. Run Phases 1 and 4 first, or supply your own files in those formats. Each question record needs `id`, `question`, `reference_answer`, `domain` and `seed_section`, and `citation` and `difficulty` are read if present; each training pair needs `question`, `domain`, and `section` or `seed_section`.

Two jobs, in order:

**Stage A: QA-filter the existing set.** Its questions were never filtered. Only a
citation regex stood between generation and the file. Phase 3 measures the
contradiction rate of the model that wrote them, so expect some of the existing set
to be flawed. Grading a fine-tune against a noisy test set makes every
number mushy. Same two judges and same rubric the training pairs got.

**Stage B: generate eval questions for the three new domains.** Phase 4 added
training pairs across Parts 112 (produce), 123 (seafood), and 120 (juice).
The existing set covers none of them, so that part of the corpus is untestable
without new questions. New questions get the same QA filter, plus a
contamination guard against every Phase 4 training pair.

**Result:** one clean test set, every item passing the same bar, all six
domains covered. The surviving originals keep their never-trained-on status.

**Authorship note:** Stage A keeps Opus's questions (filtered). Stage B uses
Opus too, for consistency of style across the set. Both halves are then held
to the same measured quality bar, which is what makes them comparable.

In [ ]:
import concurrent.futures
import json
import os
import random
import re
import statistics
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
GENERATOR = {"model": "anthropic/claude-opus-5"}
GEN_PROVIDER = {"order": ["amazon-bedrock"], "allow_fallbacks": False}

# Web search plugin for Stage B generation. Refused until its cost is bounded (see SEARCH_COST_BOUNDED).
USE_WEB_SEARCH = True
WEB_PLUGINS = [{"id": "web"}]

# Web search is refused until its cost is bounded. Set SEARCH_COST_BOUNDED True only after search uses, results and
# search-result context tokens are bounded and priced into the pre-dispatch reservation.
SEARCH_COST_BOUNDED = False


class SearchNotBounded(RuntimeError):
    pass


def request_enables_search(body):
    """True if a request body turns on any web search path: a search server tool, the web plugin, web_search_options,
    or an :online model suffix."""
    if not isinstance(body, dict):
        return False
    if any("search" in str(k).lower() for k in body) or ":online" in str(body.get("model", "")).lower():
        return True
    for t in body.get("tools") or []:
        if "search" in json.dumps(t, default=str).lower() or "web" in json.dumps(t, default=str).lower():
            return True
    for p in body.get("plugins") or []:
        if "web" in json.dumps(p, default=str).lower() or "search" in json.dumps(p, default=str).lower():
            return True
    return False


def require_bounded_search():
    if SEARCH_COST_BOUNDED is not True:
        raise SearchNotBounded(
            "Refusing to run: web search is billed separately from tokens, can run more than once per request, and "
            "adds search-result tokens, so the reservation does not bound the cost and search-enabled requests stay "
            "off. Set USE_WEB_SEARCH = False to run without search.")


JUDGES = {
    "gemini": {"slug": "google/gemini-3.1-pro-preview",
               "provider": {"order": ["google-vertex", "google-ai-studio"],
                            "allow_fallbacks": True}},
    "kimi":   {"slug": "moonshotai/kimi-k3",
               "provider": {"order": ["digitalocean"], "allow_fallbacks": False}},
}

# Stage B target: about the existing set's questions-per-section density across
# the 33 new sections.
NEW_DOMAIN_TARGET = 430      # 430 covers all 33 sections; a lower target can stop
                             # before the last juice sections
CHUNK_CHARS   = 7500
CFR_TEXT_CAP  = 15000
DEDUP_JACCARD = 0.55           # vs training pairs — same threshold Phase 4 used

GEN_WORKERS = 8
QA_WORKERS  = 32
REQUEST_TIMEOUT = 600
MAX_RETRIES = 4
MAX_TOKENS  = 32000
SEED = 20260828

VOLUME_MOUNT     = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
TRAINING_SUBDIR  = "phase4_training_pairs"
OUTPUT_SUBDIR    = "phase5_testset"

ECFR_BASE = "https://www.ecfr.gov/api/versioner/v1/full"
ECFR_DATE = "2026-08-01"
ECFR_HDRS = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Attach 'your-modal-secret'.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
           "X-OpenRouter-Title": "labs-benchmark"}
random.seed(SEED)

# Sections added in Phase 4, confirmed present in the training corpus.
NEW_SECTIONS = {
    "produce": ["112.1","112.2","112.3","112.11","112.12","112.30","112.31",
                "112.43","112.44","112.50","112.53","112.54","112.81",
                "112.111","112.112","112.140","112.141","112.143"],
    "seafood": ["123.3","123.6","123.7","123.8","123.9","123.10","123.11","123.12"],
    "juice":   ["120.3","120.6","120.7","120.8","120.11","120.12","120.14"],
}

ARCHETYPES = ["applicability", "timeline or deadline", "recordkeeping requirement",
              "exemption or qualified exemption", "definition", "procedural sequence",
              "numeric threshold", "responsible party",
              "consequence of non-compliance", "conflict between two provisions"]

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Only confirmed prices are listed; a model with no confirmed price is left out and the notebook refuses to run.
from datetime import datetime, timezone
import math


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)   # per-run cap in USD; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# USD per million tokens (input, output). Each figure is the highest listed price across the OpenRouter endpoints
# for that model (worst case), confirmed on 2026-11-05 from https://openrouter.ai/api/v1/models/<slug>/endpoints.
PRICE = {
    "anthropic/claude-opus-5": (10.0, 50.0),             # fast/priority tier
    "google/gemini-3.1-pro-preview": (7.2, 32.4),        # priority tier, prompts over 200k tokens
    "moonshotai/kimi-k3": (4.5, 22.5),                   # Parasail/Fireworks priority endpoints
}
missing_prices = sorted(set([GENERATOR["model"]] + [j["slug"] for j in JUDGES.values()]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"Refusing to run: no confirmed listed price for {missing_prices}. Add each model's confirmed "
                       "(input, output) USD per million tokens to PRICE in this cell, then rerun. Nothing was sent.")
GEN_PRICE = PRICE[GENERATOR["model"]]    # used only for the generation cost report; listed worst-case price
# None is the only signal for a new run. Any other value is a resume ID and goes through the two checks below,
# which refuse an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if not isinstance(RUN_ID, str) or not RUN_ID.strip():
    raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {RUN_ID!r}).")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_ID) or RUN_ID in (".", ".."):
    raise RuntimeError(f"RUN_ID {RUN_ID!r} is not a safe directory name. Refusing to start.")

# One ledger root and one lock for every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_LEDGER_DIR = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_LEDGER_DIR, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. One exclusive lock file at SPEND_LEDGER_DIR/runner.lock: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete that file by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_LEDGER_DIR, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be active, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print(f"Generator      : {GENERATOR['model']}")
print(f"Judges         : {list(JUDGES)}")
print(f"New sections   : {sum(len(v) for v in NEW_SECTIONS.values())}")
print(f"Stage B target : {NEW_DOMAIN_TARGET}")

## Load the existing set and the training corpus

In [ ]:
EXPECTED_EXISTING = None  # set to your Phase 1 set size to get a warning on a mismatch


def list_runs(subdir, req):
    base = os.path.join(VOLUME_MOUNT, subdir)
    return sorted(d for d in os.listdir(base)
                  if not d.startswith("_")
                  and os.path.isfile(os.path.join(base, d, req)))


qrid = list_runs(QUESTIONS_SUBDIR, "questions.jsonl")[-1]
EXISTING = [json.loads(l) for l in open(
    os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, qrid, "questions.jsonl"))]
print(f"Phase 1 run : {qrid}")
print(f"Existing questions: {len(EXISTING)}")
print(f"  by domain : {dict(Counter(q['domain'] for q in EXISTING))}")
if EXPECTED_EXISTING is not None and len(EXISTING) != EXPECTED_EXISTING:
    print(f"  *** WARNING: expected {EXPECTED_EXISTING}, got {len(EXISTING)}. "
          f"Check that this is the question set you meant to use. ***")

trid = list_runs(TRAINING_SUBDIR, "training_pairs.jsonl")[-1]
TRAIN = [json.loads(l) for l in open(
    os.path.join(VOLUME_MOUNT, TRAINING_SUBDIR, trid, "training_pairs.jsonl"))]
print(f"\nPhase 4 run : {trid}")
print(f"Training pairs on volume: {len(TRAIN):,}")
print("  NOTE: dedup runs against every pair in this file; a larger guard set is strictly safer.")
print(f"  by domain : {dict(Counter(p['domain'] for p in TRAIN))}")

## eCFR fetch

In [ ]:
def fetch_cfr_section(section, retries=4):
    part = section.split(".")[0]
    url = f"{ECFR_BASE}/{ECFR_DATE}/title-21.xml"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"part": part, "section": section},
                             timeout=60, headers=ECFR_HDRS)
            if r.status_code == 200:
                r.encoding = "utf-8"
                txt = re.sub(r"<[^>]+>", " ", r.text)
                txt = re.sub(r"\s+", " ", txt).strip()
                m = re.match(r"§\s*[\d.]+\s+(.+?\.)\s", txt)
                return (m.group(1).strip() if m else f"Section {section}"), txt
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 2))
                continue
            return None, None
        except Exception:
            time.sleep(2 ** attempt)
    return None, None


def chunk_text(section, heading, text, cap=CHUNK_CHARS):
    if len(text) <= cap:
        return [(f"{section} — {heading}", text)]
    segs = re.split(r"(?=\((?:[a-z]|[ivx]+)\)\s)", text)
    chunks, cur = [], ""
    for seg in segs:
        if len(cur) + len(seg) > cap and cur:
            chunks.append(cur.strip()); cur = seg
        else:
            cur += seg
    if cur.strip():
        chunks.append(cur.strip())
    if not chunks:
        return [(f"{section} — {heading}", text[:cap])]
    return [(f"{section} chunk {i+1}/{len(chunks)} — {heading}", c)
            for i, c in enumerate(chunks)]


def q_count(n):
    return 8 if n < 1000 else 12 if n < 2500 else 16 if n < 5000 else 20


# Sections needed: all new ones (Stage B) + every section in the existing set
# (Stage A judging needs the CFR text as ground truth).
needed = {s for secs in NEW_SECTIONS.values() for s in secs}
needed |= {q["seed_section"] for q in EXISTING}
needed = sorted(needed)

print(f"Fetching {len(needed)} sections...", flush=True)
CFR = {}
for i, sec in enumerate(needed, 1):
    if i % 15 == 0 or i == 1:
        print(f"  [{i}/{len(needed)}]", flush=True)
    h, t = fetch_cfr_section(sec)
    if t:
        CFR[sec] = {"heading": h, "text": t, "chars": len(t),
                    "chunks": chunk_text(sec, h, t)}
    time.sleep(0.25)

missing = [s for s in needed if s not in CFR]
print(f"\nRetrieved {len(CFR)}/{len(needed)}" + (f"  MISSING: {missing}" if missing else ""))

## Checkpointing

In [ ]:
class Checkpoint:
    def __init__(self, path, key_fn):
        self.path, self.key_fn = path, key_fn
        self.lock = threading.Lock()
        self.rows, self.done = [], set()
        if os.path.exists(path):
            for line in open(path):
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                self.rows.append(r); self.done.add(key_fn(r))
        self.fh = open(path, "a")

    def has(self, k):
        return k in self.done

    def add(self, row):
        with self.lock:
            self.fh.write(json.dumps(row) + "\n")
            self.fh.flush(); os.fsync(self.fh.fileno())
            self.rows.append(row); self.done.add(self.key_fn(row))

    def close(self):
        try:
            self.fh.close()
        except Exception:
            pass


CKPT_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "_checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)

qa_a_ckpt = Checkpoint(os.path.join(CKPT_DIR, "qa_existing.jsonl"),
                       lambda r: (r["judge"], r["question_id"]))
gen_ckpt  = Checkpoint(os.path.join(CKPT_DIR, "gen_new.jsonl"),
                       lambda r: r["batch_id"])
qa_b_ckpt = Checkpoint(os.path.join(CKPT_DIR, "qa_new.jsonl"),
                       lambda r: (r["judge"], r["question_id"]))

print(f"Stage A QA ckpt : {len(qa_a_ckpt.rows):,}")
print(f"Stage B gen ckpt: {len(gen_ckpt.rows):,}")
print(f"Stage B QA ckpt : {len(qa_b_ckpt.rows):,}")

## Request helper

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    if request_enables_search(body):
        require_bounded_search()
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = (len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8"))
                + len(json.dumps(body.get("response_format"), ensure_ascii=False).encode("utf-8"))
                + PROMPT_OVERHEAD_TOKENS)
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def call_or(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES, empty_retry=False):
    last = None
    for attempt in range(retries):
        try:
            r = guarded_post(body, HEADERS, timeout=timeout, purpose="call")
            if r.status_code in (429, 503):
                w = r.headers.get("Retry-After")
                time.sleep(float(w) if w and w.isdigit() else 2 ** (attempt + 2))
                last = f"HTTP {r.status_code}"; continue
            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}
                time.sleep(2 ** attempt); continue
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}
            text = (d["choices"][0]["message"].get("content") or "").strip()
            if not text and empty_retry and attempt < retries - 1:
                time.sleep(1 + attempt); continue
            u = d.get("usage", {}) or {}
            return {"ok": True, "text": text,
                    "prompt_tokens": u.get("prompt_tokens", 0),
                    "completion_tokens": u.get("completion_tokens", 0), "error": None}
        except (BudgetExceeded, SearchNotBounded):
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"; time.sleep(2 ** attempt)
    return {"ok": False, "error": last}


print("helper loaded")

## The QA rubric

Same rubric the training pairs were held to. `question_quality` matters more
here than it did there — a bad eval question corrupts the measurement rather
than just wasting a training example.

In [ ]:
QA_SYSTEM = """You review question-and-answer pairs for a food safety regulatory evaluation set. These questions will be used to grade a model, so a flawed question corrupts the measurement. You have the full text of the governing CFR section as ground truth.

question_quality (0-3): Is the question well-formed and cleanly answerable from this section?
  3=realistic and specific  2=answerable but generic  1=vague or needs outside information  0=malformed

answer_accuracy (0-5): Is the reference answer correct as a statement of this section?
  5=correct and precise  4=minor imprecision  3=one meaningful gap  2=material error  1=mostly wrong  0=wrong

pair_coherence (0-2): Does the answer answer the question asked?
  2=directly  1=partially  0=no

citation_valid (0-2): Does the citation exist in the text above and support the answer?
  2=yes  1=exists but wrong subsection  0=not in the text

contradiction (boolean): Does the answer contradict the CFR text or assert a requirement the section does not contain? Citing a related provision elsewhere is NOT a contradiction.

Return JSON only:
{"question_quality": N, "answer_accuracy": N, "pair_coherence": N, "citation_valid": N, "contradiction": true|false, "note": "one short sentence"}"""

QA_TEMPLATE = """=== GOVERNING CFR SECTION: 21 CFR {section} (ground truth) ===
{text}
=== END SECTION ===

QUESTION:
{question}

REFERENCE ANSWER:
{answer}

STATED CITATION: {citation}

Grade against the CFR text above. JSON only."""


def qa_one(jn, q, ckpt):
    j = JUDGES[jn]
    text = CFR.get(q["seed_section"], {}).get("text", "")[:CFR_TEXT_CAP]
    body = {"model": j["slug"], "provider": j["provider"], "max_tokens": 2000,
            "messages": [{"role": "system", "content": QA_SYSTEM},
                         {"role": "user", "content": QA_TEMPLATE.format(
                             section=q["seed_section"], text=text,
                             question=q["question"], answer=q["reference_answer"],
                             citation=q.get("citation", ""))}]}
    res = call_or(body, timeout=240, empty_retry=True)
    row = {"judge": jn, "question_id": q["id"], "seed_section": q["seed_section"],
           "domain": q["domain"]}
    if not res["ok"] or not res["text"]:
        row.update({"answer_accuracy": None, "error": res.get("error") or "empty"})
    else:
        try:
            txt = res["text"].replace("```json", "").replace("```", "").strip()
            b = txt.find("{"); txt = txt[b:] if b > 0 else txt
            g = json.loads(txt)
            row.update({k: int(g.get(k, 0)) for k in
                        ["question_quality", "answer_accuracy",
                         "pair_coherence", "citation_valid"]})
            row["contradiction"] = bool(g.get("contradiction"))
            row["note"] = g.get("note", "")
            row["error"] = None
        except Exception as e:
            row.update({"answer_accuracy": None,
                        "error": f"parse: {type(e).__name__} :: {res['text'][:100]}"})
    ckpt.add(row)
    return row


def run_qa(questions, ckpt, label):
    tasks = [(jn, q) for jn in JUDGES for q in questions
             if not ckpt.has((jn, q["id"]))]
    print(f"{label}: {len(JUDGES)*len(questions):,} gradings, {len(tasks):,} remaining",
          flush=True)
    t0 = time.time(); n = [0]

    def _t(a):
        r = qa_one(a[0], a[1], ckpt)
        n[0] += 1
        if n[0] % 200 == 0:
            el = time.time() - t0
            print(f"  {n[0]:,}/{len(tasks):,}  eta {(len(tasks)-n[0])/(n[0]/el)/60:.0f}m",
                  flush=True)
        return r

    if tasks:
        with concurrent.futures.ThreadPoolExecutor(max_workers=QA_WORKERS) as pool:
            list(pool.map(_t, tasks))
    return time.time() - t0

## Drop rule

Stricter than Phase 4's, because a bad eval question corrupts measurement
rather than diluting training. Notably: **a missing judge is not a pass.**
In Phase 4 that bypass lets ungraded pairs into the output.

In [ ]:
def apply_drop_rule(questions, grades):
    pg = defaultdict(dict)
    for g in grades:
        if g.get("answer_accuracy") is not None:
            pg[g["question_id"]][g["judge"]] = g

    keep, drop = [], []
    for q in questions:
        gm = pg.get(q["id"], {})
        if len(gm) < len(JUDGES):
            q["_drop_reasons"] = ["incomplete_grades"]
            drop.append(q); continue
        gs = list(gm.values())
        reasons = []
        if any(v["answer_accuracy"] <= 3 for v in gs):
            reasons.append("accuracy<=3")
        if any(v["question_quality"] <= 1 for v in gs):
            reasons.append("question_quality<=1")
        if any(v["citation_valid"] == 0 for v in gs):
            reasons.append("citation=0")
        if any(v["pair_coherence"] == 0 for v in gs):
            reasons.append("coherence=0")
        if any(v["contradiction"] for v in gs):
            reasons.append("contradiction")
        for dim in ["question_quality", "answer_accuracy",
                    "pair_coherence", "citation_valid"]:
            q[f"qa_{dim}"] = round(statistics.mean(v[dim] for v in gs), 2)
        q["qa_contradiction_flags"] = sum(1 for v in gs if v["contradiction"])
        if reasons:
            q["_drop_reasons"] = reasons
            drop.append(q)
        else:
            keep.append(q)
    return keep, drop


print("Drop rule (any judge triggers):")
print("  accuracy <= 3, question_quality <= 1, citation = 0,")
print("  coherence = 0, contradiction flagged, or grades incomplete")

## Stage A: filter the existing set

In [ ]:
elapsed_a = run_qa(EXISTING, qa_a_ckpt, "Stage A QA")
GRADES_A = qa_a_ckpt.rows
ok_a = [g for g in GRADES_A if g.get("answer_accuracy") is not None]
print(f"\nStage A: {elapsed_a/60:.1f} min | parsed {len(ok_a):,}/{len(GRADES_A):,}")

KEEP_A, DROP_A = apply_drop_rule(EXISTING, GRADES_A)
print(f"\nExisting -> {len(KEEP_A)} kept, {len(DROP_A)} dropped "
      f"({len(DROP_A)/max(len(EXISTING),1):.1%})")
print(f"  drop reasons: {dict(Counter(r for q in DROP_A for r in q['_drop_reasons']))}")
print(f"  kept by domain: {dict(Counter(q['domain'] for q in KEEP_A))}")
if KEEP_A:
    print(f"  kept mean accuracy: "
          f"{statistics.mean(q['qa_answer_accuracy'] for q in KEEP_A):.3f}/5")

contra_rate = sum(1 for g in ok_a if g.get("contradiction")) / max(len(ok_a), 1)
print(f"\nMeasured contradiction rate on the original set: {contra_rate:.1%}")
print(f"  (compare with the contradiction rate from your Phase 3 run)")

## Stage B — generate new-domain questions

In [ ]:
GEN_SYSTEM = """You are building an evaluation set for a food safety regulatory model. These questions will score a fine-tuned model, so a flawed question corrupts the measurement. Precision matters more than volume.

The full text of the governing CFR section is provided. Write questions and answers from that text. Do not rely on memory."""

GEN_TEMPLATE = """Generate up to {n} questions.

=== GOVERNING SECTION: {label} ===
{text}
=== END SECTION ===

ARCHETYPE for this batch: {archetype}

Each question must:
- Be answerable from the section text above
- Be open-ended. No multiple choice, no yes/no
- Describe a real situation a food safety professional would face
- Use practitioner language, not regulatory boilerplate
- Be distinct from the others in this batch

Each reference answer must:
- Cite the specific subsection, e.g. "21 CFR {section}(b)(2)"
- Track the operative language from the text closely
- State the requirement plainly in under 120 words
- Note any condition or exception that changes the answer
- Contain nothing that is not in the section text above

Difficulty mix: roughly 30% straightforward recall, 50% requiring application to the scenario, 20% requiring reconciliation of two or more provisions.

If the section does not support {n} distinct questions of this archetype, return fewer. Do not pad. Every citation must appear in the text above.

Return JSON only, matching the schema."""

GEN_SCHEMA = {"type": "json_schema", "json_schema": {"name": "eval_batch", "strict": True,
    "schema": {"type": "object", "properties": {"questions": {"type": "array", "items": {
        "type": "object",
        "properties": {"question": {"type": "string"},
                       "reference_answer": {"type": "string"},
                       "citation": {"type": "string"},
                       "difficulty": {"type": "string",
                                      "enum": ["recall", "application", "reconciliation"]}},
        "required": ["question", "reference_answer", "citation", "difficulty"],
        "additionalProperties": False}}},
        "required": ["questions"], "additionalProperties": False}}}

CIT_RE = re.compile(r"21\s*CFR\s*\d+[\.\d]+(?:\([a-z0-9]+\))*")


def norm_cite(q):
    c = q.get("citation")
    if isinstance(c, str) and c.strip():
        return c
    cs = q.get("citations")
    if isinstance(cs, list) and cs:
        return "; ".join(str(x) for x in cs)
    if isinstance(cs, str):
        return cs
    m = CIT_RE.search(q.get("reference_answer", ""))
    return m.group(0) if m else ""


# Seeds: every section gets a DIFFERENT archetype as we walk the list, so a
# target reached partway through the first pass still yields archetype variety.
# (Advancing the archetype only after a full pass would give one archetype
# whenever the target is reached inside the first pass.)
flat = [(dom, sec) for dom, secs in NEW_SECTIONS.items() for sec in secs
        if sec in CFR]
if not flat:
    raise RuntimeError("No new-domain sections retrieved from eCFR.")

seeds, projected, si = [], 0, 0
max_seeds = len(flat) * len(ARCHETYPES)
while projected < NEW_DOMAIN_TARGET and si < max_seeds:
    dom, sec = flat[si % len(flat)]
    v = CFR[sec]
    lap = si // len(flat)                       # which time around the sections
    ci = lap % len(v["chunks"])                 # rotate chunks on later laps
    ai = (si + lap) % len(ARCHETYPES)           # rotate archetype per seed
    label, chunk = v["chunks"][ci]
    ask = q_count(len(chunk))
    seeds.append({"batch_id": f"{dom[:3]}-{sec.replace('.','')}-a{ai:02d}-c{ci}",
                  "domain": dom, "section": sec, "label": label, "text": chunk,
                  "archetype": ARCHETYPES[ai], "ask": ask})
    projected += ask
    si += 1

print(f"Stage B seeds: {len(seeds)} batches, projected ~{projected} questions")
print(f"  by domain    : {dict(Counter(s['domain'] for s in seeds))}")
print(f"  sections hit : {len(set(s['section'] for s in seeds))}/{len(flat)}")
print(f"  archetypes   : {len(set(s['archetype'] for s in seeds))}/{len(ARCHETYPES)}")
if len(set(s["archetype"] for s in seeds)) < 5:
    print("  *** WARNING: low archetype variety. Raise NEW_DOMAIN_TARGET. ***")


def generate_batch(seed):
    body = {"model": GENERATOR["model"], "provider": GEN_PROVIDER,
            "max_tokens": MAX_TOKENS, "response_format": GEN_SCHEMA,
            "messages": [{"role": "system", "content": GEN_SYSTEM},
                         {"role": "user", "content": GEN_TEMPLATE.format(
                             n=seed["ask"], label=seed["label"], text=seed["text"],
                             archetype=seed["archetype"], section=seed["section"])}]}
    if USE_WEB_SEARCH:
        require_bounded_search()
        body["plugins"] = WEB_PLUGINS
    res = call_or(body)
    row = {"batch_id": seed["batch_id"], "section": seed["section"],
           "domain": seed["domain"], "archetype": seed["archetype"],
           "asked": seed["ask"], "ok": res["ok"], "error": res.get("error"),
           "prompt_tokens": res.get("prompt_tokens", 0),
           "completion_tokens": res.get("completion_tokens", 0), "questions": []}
    if res["ok"]:
        try:
            txt = res["text"].replace("```json", "").replace("```", "").strip()
            b = txt.find("{"); txt = txt[b:] if b > 0 else txt
            for i, q in enumerate(json.loads(txt).get("questions", [])):
                cite = norm_cite(q)
                if not CIT_RE.search(cite) or seed["section"] not in cite:
                    continue
                if len(q.get("question", "")) < 20:
                    continue
                row["questions"].append({
                    "id": f"{seed['batch_id']}-{i+1:02d}",
                    "question": q["question"],
                    "reference_answer": q["reference_answer"],
                    "citation": cite,
                    "difficulty": q.get("difficulty", "application"),
                    "seed_section": seed["section"], "domain": seed["domain"],
                    "batch_id": seed["batch_id"],
                    "seed_archetype": seed["archetype"]})
        except Exception as e:
            row["error"] = f"parse: {type(e).__name__}"
    gen_ckpt.add(row)
    return row


todo = [s for s in seeds if not gen_ckpt.has(s["batch_id"])]
print(f"\nGenerating {len(todo)} batches (done: {len(seeds)-len(todo)})", flush=True)
_t0 = time.time(); _n = [0]


def _tg(s):
    r = generate_batch(s)
    _n[0] += 1
    if _n[0] % 10 == 0:
        got = sum(len(b["questions"]) for b in gen_ckpt.rows)
        print(f"  {_n[0]}/{len(todo)} batches, {got} questions", flush=True)
    return r


if todo:
    with concurrent.futures.ThreadPoolExecutor(max_workers=GEN_WORKERS) as pool:
        list(pool.map(_tg, todo))
gen_elapsed = time.time() - _t0

NEW_RAW = [q for b in gen_ckpt.rows for q in b["questions"]]
gen_cost = sum(b["prompt_tokens"] / 1e6 * GEN_PRICE[0]
               + b["completion_tokens"] / 1e6 * GEN_PRICE[1] for b in gen_ckpt.rows)
print(f"\nGenerated {len(NEW_RAW)} raw questions in {gen_elapsed/60:.1f} min  (${gen_cost:.2f})")
print(f"  by domain: {dict(Counter(q['domain'] for q in NEW_RAW))}")

## Contamination guard

Every new eval question is checked against all training pairs. Anything too
close gets dropped — the model must not be graded on what it trained on.

In [ ]:
STOP = set("the a an of to in for and or is are be if what when must may you "
           "your this that with from on at as it its under section cfr does do "
           "can shall will not no any all each".split())


def cw(s):
    return frozenset(w for w in re.findall(r"[a-z]{4,}", s.lower()) if w not in STOP)


train_fp_by_sec = defaultdict(list)
for p in TRAIN:
    train_fp_by_sec[p.get("section", p.get("seed_section", ""))].append(cw(p["question"]))

NEW_CLEAN, contaminated = [], []
for q in NEW_RAW:
    c = cw(q["question"])
    fps = train_fp_by_sec.get(q["seed_section"], [])
    if any(len(c & fp) / len(c | fp) >= DEDUP_JACCARD for fp in fps if c | fp):
        contaminated.append(q)
    else:
        NEW_CLEAN.append(q)

print(f"Contamination check against {len(TRAIN):,} training pairs:")
print(f"  clean       : {len(NEW_CLEAN)}")
print(f"  contaminated: {len(contaminated)} dropped")

# also dedup within the new questions themselves
seen_by_sec, deduped, internal_dupes = defaultdict(list), [], 0
for q in NEW_CLEAN:
    c = cw(q["question"])
    if any(len(c & fp) / len(c | fp) >= DEDUP_JACCARD
           for fp in seen_by_sec[q["seed_section"]] if c | fp):
        internal_dupes += 1
        continue
    seen_by_sec[q["seed_section"]].append(c)
    deduped.append(q)
NEW_CLEAN = deduped
print(f"  internal dupes dropped: {internal_dupes}")
print(f"  into QA: {len(NEW_CLEAN)}")

## Stage B QA

In [ ]:
elapsed_b = run_qa(NEW_CLEAN, qa_b_ckpt, "Stage B QA")
GRADES_B = qa_b_ckpt.rows
ok_b = [g for g in GRADES_B if g.get("answer_accuracy") is not None]
print(f"\nStage B: {elapsed_b/60:.1f} min | parsed {len(ok_b):,}/{len(GRADES_B):,}")

KEEP_B, DROP_B = apply_drop_rule(NEW_CLEAN, GRADES_B)
print(f"\nNew questions -> {len(KEEP_B)} kept, {len(DROP_B)} dropped "
      f"({len(DROP_B)/max(len(NEW_CLEAN),1):.1%})")
print(f"  drop reasons: {dict(Counter(r for q in DROP_B for r in q['_drop_reasons']))}")
print(f"  kept by domain: {dict(Counter(q['domain'] for q in KEEP_B))}")

## Merge and write

In [ ]:
for q in KEEP_A:
    q["origin"] = "phase1_filtered"
for q in KEEP_B:
    q["origin"] = "phase5_new_domains"
TESTSET = KEEP_A + KEEP_B

run_id = RUN_ID
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)


def wj(name, rows):
    p = os.path.join(outdir, name)
    with open(p, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    return p


print("=" * 74)
print(f"FINAL TEST SET: {len(TESTSET):,} questions")
print("=" * 74)
print(f"  from existing set : {len(KEEP_A)}  ({len(DROP_A)} dropped)")
print(f"  new domains       : {len(KEEP_B)}")
print(f"\n  by domain     : {dict(Counter(q['domain'] for q in TESTSET))}")
print(f"  by difficulty : {dict(Counter(q.get('difficulty') for q in TESTSET))}")
print(f"  sections      : {len(set(q['seed_section'] for q in TESTSET))}")
print(f"  mean accuracy : "
      f"{statistics.mean(q['qa_answer_accuracy'] for q in TESTSET):.3f}/5")
print(f"  mean q-quality: "
      f"{statistics.mean(q['qa_question_quality'] for q in TESTSET):.3f}/3")

manifest = {
    "run_id": run_id, "stage": "5_testset_rebuild",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "option": "filter existing + add new domains",
    "source_question_run": qrid, "source_training_run": trid,
    "generator": GENERATOR["model"], "qa_judges": list(JUDGES),
    "drop_rule": ["any accuracy<=3", "any question_quality<=1", "any citation=0",
                  "any coherence=0", "any contradiction", "incomplete grades"],
    "stage_a": {"input": len(EXISTING), "kept": len(KEEP_A), "dropped": len(DROP_A),
                "drop_reasons": dict(Counter(r for q in DROP_A
                                             for r in q["_drop_reasons"])),
                "measured_contradiction_rate": round(contra_rate, 4)},
    "stage_b": {"generated": len(NEW_RAW), "contaminated_dropped": len(contaminated),
                "internal_dupes_dropped": internal_dupes,
                "into_qa": len(NEW_CLEAN), "kept": len(KEEP_B), "dropped": len(DROP_B),
                "drop_reasons": dict(Counter(r for q in DROP_B
                                             for r in q["_drop_reasons"]))},
    "final": {"total": len(TESTSET),
              "by_domain": dict(Counter(q["domain"] for q in TESTSET)),
              "by_difficulty": dict(Counter(q.get("difficulty") for q in TESTSET)),
              "sections": len(set(q["seed_section"] for q in TESTSET)),
              "mean_accuracy": round(statistics.mean(
                  q["qa_answer_accuracy"] for q in TESTSET), 3)},
    "generation_cost_usd": round(gen_cost, 2),
    "elapsed_min": {"qa_existing": round(elapsed_a / 60, 1),
                    "generate": round(gen_elapsed / 60, 1),
                    "qa_new": round(elapsed_b / 60, 1)},
    "files": {"testset": wj("testset.jsonl", TESTSET),
              "dropped_existing": wj("dropped_existing.jsonl", DROP_A),
              "dropped_new": wj("dropped_new.jsonl", DROP_B),
              "contaminated": wj("contaminated.jsonl", contaminated),
              "qa_grades_existing": wj("qa_grades_existing.jsonl", ok_a),
              "qa_grades_new": wj("qa_grades_new.jsonl", ok_b)},
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

qa_a_ckpt.close(); gen_ckpt.close(); qa_b_ckpt.close()

print(f"\nGeneration cost: ${manifest['generation_cost_usd']:.2f}")
print(f"Output: {outdir}")
print(f"\ntestset.jsonl -> {len(TESTSET):,} questions, all six domains")

## Inspect

Read what got dropped from the existing set before accepting the cut. If the
drops look like judge errors rather than real defects, loosen the rule and
re-run — the grades are checkpointed, so re-filtering costs nothing.

In [ ]:
def show(qs, n=4, label=""):
    print(f"\n{label}\n" + "=" * 74)
    for q in qs[:n]:
        print(f"{q['id']}  21 CFR {q['seed_section']}  {q['domain']}")
        print(f"  reasons: {q.get('_drop_reasons')}")
        print(f"  scores : qq={q.get('qa_question_quality')} "
              f"acc={q.get('qa_answer_accuracy')} "
              f"coh={q.get('qa_pair_coherence')} "
              f"cite={q.get('qa_citation_valid')} "
              f"contra_flags={q.get('qa_contradiction_flags')}")
        print(f"  Q: {q['question'][:220]}")
        print(f"  A: {q['reference_answer'][:260]}\n")


if DROP_A:
    show(DROP_A, 4, "DROPPED FROM THE EXISTING SET")
if DROP_B:
    show(DROP_B, 3, "DROPPED FROM NEW DOMAINS")
if contaminated:
    print("\nCONTAMINATED (too close to a training pair)\n" + "=" * 74)
    for q in contaminated[:3]:
        print(f"{q['id']}  {q['seed_section']}")
        print(f"  Q: {q['question'][:220]}\n")